## This manuscript contains an analysis of football matches since 2000

At first, we load the dataset using pandas

In [6]:
import pandas as pd
raw_df = pd.read_csv('data/Matches.csv')

/tmp/ipykernel_45899/1280493266.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('data/Matches.csv')


In [8]:
raw_df.dtypes

Division        object
MatchDate       object
MatchTime       object
HomeTeam        object
AwayTeam        object
HomeElo        float64
AwayElo        float64
Form3Home      float64
Form5Home      float64
Form3Away      float64
Form5Away      float64
FTHome         float64
FTAway         float64
FTResult        object
HTHome         float64
HTAway         float64
HTResult        object
HomeShots      float64
AwayShots      float64
HomeTarget     float64
AwayTarget     float64
HomeFouls      float64
AwayFouls      float64
HomeCorners    float64
AwayCorners    float64
HomeYellow     float64
AwayYellow     float64
HomeRed        float64
AwayRed        float64
OddHome        float64
OddDraw        float64
OddAway        float64
MaxHome        float64
MaxDraw        float64
MaxAway        float64
Over25         float64
Under25        float64
MaxOver25      float64
MaxUnder25     float64
HandiSize      float64
HandiHome      float64
HandiAway      float64
C_LTH          float64
C_LTA      

In [9]:
raw_df.head(3)

,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,...,MaxUnder25,HandiSize,HandiHome,HandiAway,C_LTH,C_LTA,C_VHD,C_VAD,C_HTB,C_PHB
0,F1,2000-07-28,NaN,Marseille,Troyes,1686.34,1586.57,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F1,2000-07-28,NaN,Paris SG,Strasbourg,1714.89,1642.51,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2000-07-28,NaN,Wasquehal,Nancy,1465.08,1633.80,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Formatting

The dataset already features built-in features that are taking away all the work. Therefore the lagged columns, the pre-computed likelihoods and other handy values from betting companies etc. will be removed:

In [13]:
keep_columns = [
    'Division',
    'MatchDate',
    'MatchTime',
    'HomeTeam',
    'AwayTeam',
    'FTHome',
    'FTAway',
    'FTResult',
    'HTHome',
    'HTAway',
    'HTResult',
    'HomeShots',
    'AwayShots',
    'HomeTarget',
    'AwayTarget',
    'HomeCorners',
    'AwayCorners',
    'HomeYellow',
    "AwayYellow",
    'HomeRed',
    'AwayRed',
]

df = raw_df[keep_columns]

## Merging the elos from EloRatings.csv

In [50]:
# read the elos
elos_df = pd.read_csv("data/EloRatings.csv")

# inspect columns
elos_df.dtypes

date        object
club        object
country     object
elo        float64
dtype: object

In [52]:
elos_df.isna().sum()

date       0
club       0
country    0
elo        0
dtype: int64

In [65]:
elos_df.head()

,date,club,country,elo
0,2000-07-01,Aachen,GER,1453.60
1,2000-07-01,Aalborg,DEN,1482.61
2,2000-07-01,Aalst,BEL,1337.53
3,2000-07-01,Aarhus,DEN,1381.46
4,2000-07-01,Aberdeen,SCO,1360.43


Joining the datasets is not as easy as standard joins. There are several problems to watch out for:
1. Could there be several teams with the same names? We need to check that.
2. The elo timestamps are probably not given for every matchday. Therefore we need to join using the closest previous rating (since we don't want to use knowledge from the future; when we use the timestamp from the future we might 'cheat' by inferring that the elo went up after a match for exampl)

In [66]:
# check if there are clubs with the same name in different countries
clubs = elos_df['club'].unique()
for c in clubs:
    countries = elos_df[elos_df['club'] == c]['country'].unique()
    if len(countries) > 1:
        print(f"Double occurence of {c}, present in countries {countries}")

## this was the naive way to implement this (very inefficient), it takes around 25 seconds to execute on a laptop

In [67]:
# we use the groupby function for more efficiency, it groups the rows by club
# then we check the unique countries for each 'club group'
club_countries = elos_df.groupby("club")["country"].unique()

# duplicates have more than one country assigned
duplicates = club_countries[club_countries.str.len() > 1]

for club, countries in duplicates.items():
    print(f"Double occurrence of {club}, present in countries {countries}")

## this only takes 0.1 seconds

In [68]:
# now we merge the datasets
# TODO: DO THE MERGE, THE IDEA IS TO COPY THE ELO OF EACH CLUB TO THE MATCH ROW FROM THE LATEST RAKING !BEFORE! THE MATCH

## Handling Missing Values

We now inspect the given data for missing values

In [14]:
df.isna().sum()

Division            0
MatchDate           0
MatchTime      131485
HomeTeam            0
AwayTeam            0
FTHome              3
FTAway              3
FTResult            3
HTHome          54580
HTAway          54580
HTResult        54580
HomeShots      115822
AwayShots      115819
HomeTarget     116628
AwayTarget     116625
HomeCorners    116194
AwayCorners    116194
HomeYellow     111259
AwayYellow     111258
HomeRed        111258
AwayRed        111260
dtype: int64

In [20]:
# inspect 3 matches where no result is given. For other missing values,
# we can still infer something from the final result
condition = df['FTResult'].isna()
df[condition]

,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,FTHome,FTAway,FTResult,HTHome,HTAway,...,HomeShots,AwayShots,HomeTarget,AwayTarget,HomeCorners,AwayCorners,HomeYellow,AwayYellow,HomeRed,AwayRed
112580,G1,2015-05-10,NaN,Niki Volos,OFI,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130377,BRA,2016-11-12,19:00:00,Chapecoense-SC,Atletico-MG,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159189,G1,2019-03-17,NaN,Panathinaikos,Olympiakos,NaN,NaN,NaN,0.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# remove these columns since they don't contain any important information and it is also only a tiny fraction of all data
if condition.any():
    df = df[~condition]
df['FTResult'].isna().sum()

/tmp/ipykernel_45899/2833033743.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~condition]


np.int64(0)

In [ ]:
# TODO: handle other missing values, imputation or discard???

In [70]:
# TODO: do analysis like what conditions predict winning something, how different teams develop, analyze trends and so on, also try to build a model that predicts the winner of a match